# LG-PM major revision — reproducible re-analyses (run once, ~2 h on T4)

Every script first **reproduces the locked published numbers** (lines marked `[GATE OK]`) and
stops with `[GATE FAIL]` if it cannot, so no new number is ever computed from the wrong
artifacts. Outputs go to `/kaggle/working/revision_outputs/` and are zipped at the end.

**Settings:** Accelerator = GPU T4 x2 · Internet = On (for `git clone`; or attach the repo as a dataset instead).

**Attach these datasets (Add Input):**

| Dataset | Used by |
|---|---|
| TN3K_Data (contains DDTI) | s02, s03, s04 |
| TNUI-2021 | s03 |
| TN5000 | s02 |
| ThyUS2Path | s01 |
| Thyroid_phase1_results (`best_resnet34_seed42.pt`) | s01–s03 |
| phase2 prototype seg (`best_fusion_decoder.pt`) | s02, s03 |
| Phase_3_classification | s01, s02, s05 |
| phase3_c_ | s01, s02, s05 |
| phase3d_external, phase3e_external | s05 |
| phase4_joint | s01, s05 |
| phase5_thyus2path_v2 (or v1 — scores are identical) | s01, s05 |
| phase6_xai | s02 |
| DDTI: Thyroid Ultrasound Images (only if Phase 3e used it) | s04 |

In [ ]:
REPO_URL = "https://github.com/YOUR-ACCOUNT/lgpm-revision.git"   # <- edit after you push
import os, subprocess
if not os.path.exists("/kaggle/working/lgpm-revision"):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/kaggle/working/lgpm-revision"], check=True)
os.chdir("/kaggle/working/lgpm-revision")
!git log -1 --oneline
!nvidia-smi --query-gpu=name,memory.total --format=csv
!ls /kaggle/input/datasets/*/ 2>/dev/null || ls /kaggle/input

## 1. DDTI label provenance (CPU, ~2 min) — R2.7

In [ ]:
!python scripts/s04_ddti_label_profile.py 2>&1 | tail -n 60

## 2. CPU re-analyses from saved predictions (~3 min) — R2.2, R2.3, R2.5

In [ ]:
!python scripts/s05_cpu_analyses.py 2>&1 | grep -E 'GATE|failed|Error|done' 

## 3. ThyUS2Path operating points for all six models (GPU, ~15 min) — R2.2, R1.2

In [ ]:
!python scripts/s01_thyus2path_all_models.py 2>&1 | tail -n 80

## 4. Explanation controls on TN5000-test (GPU, ~10 min) — R2.4, R1.7

In [ ]:
!python scripts/s02_xai_controls.py 2>&1 | tail -n 60

## 5. Support-free decoder control (GPU, ~60–80 min) — R2.1, R1.3

In [ ]:
!python scripts/s03_support_free_decoder.py 2>&1 | tail -n 80

## 6. Package the outputs — download `revision_outputs.zip` from the Output panel and send it back

In [ ]:
!cd /kaggle/working && rm -f revision_outputs.zip && zip -qr revision_outputs.zip revision_outputs -x '*.pt'
!ls -la /kaggle/working/revision_outputs/*/ && du -sh /kaggle/working/revision_outputs.zip
!grep -h 'GATE' /kaggle/working/revision_outputs/*/log.txt